# Extracción de población por Borough mediante Web Scraping

Este notebook presenta el proceso utilizado para obtener la población de los cinco
boroughs de Nueva York a partir de información publicada por el New York State
Department of Health.

**Dependencias:** `requests`, `beautifulsoup4`, `pandas`, `pytesseract` y `Pillow`.

## 1. Importación de librerías

Se utilizan `Requests` y `BeautifulSoup` para acceder e inspeccionar el HTML,
`Pytesseract` y `Pillow` para el procesamiento OCR de la imagen, y `Pandas`
para la transformación de los datos obtenidos.

In [12]:
import re
import requests
import pandas as pd
import pytesseract

from bs4 import BeautifulSoup
from PIL import Image

## 2. Obtención de la fuente

La fuente utilizada corresponde al New York State Department of Health, específicamente
a las estimaciones de población del New York State Cancer Registry.

In [13]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/140.0.0.0 Safari/537.36"
    )
}

url = "https://www.health.ny.gov/statistics/cancer/registry/population.htm"
image_path = "graphs/populationTable.png"

### Localización y descarga de la imagen

Durante la inspección del código HTML se identificó que la visualización de
Tableau referencia una imagen PNG que contiene la tabla de población. Esta
imagen se utiliza como fuente para el proceso posterior de OCR.

In [14]:
response = requests.get(url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

image_url = ""

for param in soup.find_all("param"):
    src = param.get("value")

    if src and src.endswith(".png"):
        image_url = src
        break

print(f"Imagen encontrada: {image_url}")

Imagen encontrada: https://public.tableau.com/static/images/ca/cancerstatisticreportcounty/12_Countypopulation/1.png


In [15]:
image_response = requests.get(image_url)
image_response.raise_for_status()

with open(image_path, "wb") as f:
    f.write(image_response.content)

print(f"Imagen guardada en: {image_path}")

Imagen guardada en: graphs/populationTable.png


## 3. Extracción de información mediante OCR

Debido a que la información no se encuentra disponible como una tabla HTML
estructurada, se aplica reconocimiento óptico de caracteres (OCR) sobre la
imagen descargada.

Además del texto reconocido, se conservan las coordenadas espaciales de cada
elemento para posteriormente reconstruir la estructura de la tabla.

In [16]:
img = Image.open(image_path)

data = pytesseract.image_to_data(
    img,
    lang="eng",
    config="--psm 6",
    output_type=pytesseract.Output.DICT
)

df_ocr = pd.DataFrame(data)

df_ocr = df_ocr[
    df_ocr["text"].fillna("").str.strip() != ""
].copy()

df_ocr[["text", "left", "top", "width", "height", "conf"]].head()

,text,left,top,width,height,conf
4,New,257,46,70,25,96
5,York,336,45,72,27,93
6,County/NYC,419,42,191,37,92
7,Neighborhood,623,45,229,33,96
8,Cancer,865,46,109,26,96


## 4. Preparación de la información

La imagen contiene dos tablas principales: una correspondiente a Counties/NYC
y otra correspondiente a PUMA. Como el objetivo es obtener la población de los
cinco boroughs, se utiliza únicamente la tabla de Counties.

La separación se realiza utilizando la segunda aparición de la etiqueta
`Average`, que marca el inicio de la sección PUMA.

In [17]:
average_rows = df_ocr[
    df_ocr["text"].str.strip().str.lower() == "average"
].sort_values("top")

puma_start_y = average_rows.iloc[1]["top"]

county_ocr = df_ocr[
    df_ocr["top"] < puma_start_y
].copy()

print(f"Inicio de PUMA: {puma_start_y}")
print(f"Elementos en tabla de Counties: {len(county_ocr)}")

Inicio de PUMA: 1567
Elementos en tabla de Counties: 298


## 5. Identificación de los Boroughs

La fuente presenta la información utilizando los nombres de los condados
correspondientes a cada borough. Por esta razón, se establece la siguiente
correspondencia:

| Borough | County |
|---|---|
| Bronx | Bronx |
| Brooklyn | Kings |
| Manhattan | New York |
| Queens | Queens |
| Staten Island | Richmond |

El caso de `New York` requiere un tratamiento particular, ya que el OCR
reconoce `New` y `York` como elementos independientes.

In [18]:
new_york_top = None

new_rows = county_ocr[
    county_ocr["text"].str.strip().str.lower() == "new"
].copy()

for _, new_row in new_rows.iterrows():

    y = new_row["top"]

    same_line = county_ocr[
        ((county_ocr["top"] - y).abs() <= 5) &
        (county_ocr["left"] < 250)
    ].sort_values("left")

    words = same_line["text"].str.strip().tolist()

    if words == ["New", "York"]:
        new_york_top = y
        break

In [19]:
county_tops = {
    "Bronx": None,
    "Kings": None,
    "New York": new_york_top,
    "Queens": None,
    "Richmond": None
}

for county in ["Bronx", "Kings", "Queens", "Richmond"]:

    row = county_ocr[
        county_ocr["text"].str.strip().str.lower() == county.lower()
    ]

    county_tops[county] = row.iloc[0]["top"]

county_tops

{'Bronx': np.int64(369),
 'Kings': np.int64(777),
 'New York': 925,
 'Queens': np.int64(1116),
 'Richmond': np.int64(1155)}

## 6. Extracción de población

Se definen funciones auxiliares para convertir los valores reconocidos por OCR
a números y para identificar las tres columnas de población correspondientes a
cada condado.

In [20]:
def to_int(text):
    return int(re.sub(r"\D", "", text))


def get_population_row(county_top):

    row = county_ocr[
        (county_ocr["top"] - county_top).abs() <= 15
    ].copy()

    female = row[
        (row["left"] >= 550) &
        (row["left"] < 800)
    ]

    male = row[
        (row["left"] >= 1000) &
        (row["left"] < 1200)
    ]

    total = row[
        (row["left"] >= 1450) &
        (row["left"] < 1650)
    ]

    for _, f in female.iterrows():
        for _, m in male.iterrows():
            for _, t in total.iterrows():

                f_value = to_int(f["text"])
                m_value = to_int(m["text"])
                t_value = to_int(t["text"])

                difference = abs(f_value + m_value - t_value)

                if difference <= 1:
                    return {
                        "Female": f["text"],
                        "Male": m["text"],
                        "Male and Female": t["text"]
                    }

    return None

## 7. Resultado final

Finalmente, los nombres de los condados se transforman a sus respectivos
boroughs y los valores de población se convierten a variables numéricas.

In [21]:
population_data = []

for county, top in county_tops.items():

    values = get_population_row(top)

    population_data.append({
        "County": county,
        "Female": values["Female"],
        "Male": values["Male"],
        "Total": values["Male and Female"]
    })

population_df = pd.DataFrame(population_data)


In [22]:
for col in ["Female", "Male", "Total"]:
    population_df[col] = (
        population_df[col]
        .str.replace(",", "", regex=False)
        .astype(int)
    )


In [23]:
borough_map = {
    "Bronx": "Bronx",
    "Kings": "Brooklyn",
    "New York": "Manhattan",
    "Queens": "Queens",
    "Richmond": "Staten Island"
}

population_df["Borough"] = population_df["County"].map(borough_map)

population_df = population_df[
    ["Borough", "Female", "Male", "Total"]
]


population_df.to_csv("/opt/cluster/proyecto_ny/data/raw/population_boroughs.csv", index=False)

population_df

,Borough,Female,Male,Total
0,Bronx,750962,672768,1423729
1,Brooklyn,1390435,1265135,2655570
2,Manhattan,855829,782195,1638025
3,Queens,1197626,1144421,2342047
4,Staten Island,252311,241997,494308
